# Notebook 7: ROC Curve & AUC

**Difficulty:** ⭐⭐⭐ (3 / 5)  
**Estimated time:** 2.5 hours  
**Theme:** Credit Card Fraud Detection  

---

## Why Does This Matter?

When your fraud-detection model outputs a probability score, you need to decide: _at what score do we flag a transaction as fraud?_  
Every possible answer to that question (every **threshold**) gives a different tradeoff between:
- catching real fraud (good), and
- falsely blocking legitimate purchases (bad — angry customers).

The **ROC curve** plots ALL of those tradeoffs at once, and **AUC** (Area Under the Curve) collapses the whole picture into a single number you can use to compare models.

---

## Real-World Analogy — The Airport Metal Detector

Imagine you are the engineer calibrating the **sensitivity** of an airport metal detector.

| Sensitivity setting | Effect |
|---|---|
| Very HIGH | Catches every weapon (high TPR) — but also beeps for belt buckles, coins, and keys (high FPR) |
| Very LOW | Almost no false alarms (low FPR) — but may miss real threats (low TPR) |
| Just right | A balanced operating point chosen for your airport's risk tolerance |

The ROC curve is the **full dial of possible sensitivity settings** drawn as a curve. Each point on the curve is one setting. AUC summarises how good the detector hardware (model) is — independently of which setting you choose.

---

## Historical Note

ROC stands for **Receiver Operating Characteristic** — a term coined by radar engineers during World War II when they needed to separate enemy aircraft signals from noise. The same mathematics now powers fraud detection, medical diagnosis, and spam filters.

## Plain-English Concept

**Step 1 — Your model outputs a probability, not a hard label.**  
Example: transaction A gets score 0.92 (likely fraud), transaction B gets 0.31 (probably legit).

**Step 2 — You pick a threshold.**  
If `score >= threshold` → predict FRAUD. Otherwise → predict LEGIT.

**Step 3 — Each threshold produces a confusion matrix.**  
From that matrix you can compute:  

$$\text{TPR (Recall)} = \frac{TP}{TP + FN} \quad \text{(fraction of actual fraud caught)}$$

$$\text{FPR} = \frac{FP}{FP + TN} \quad \text{(fraction of legit transactions wrongly flagged)}$$

**Step 4 — Sweep every possible threshold (0 → 1) and plot (FPR, TPR).**  
That is your ROC curve.

**Key landmarks:**
- Point `(0, 0)`: threshold = 1.0, predict nothing as fraud → TPR=0, FPR=0  
- Point `(1, 1)`: threshold = 0.0, predict everything as fraud → TPR=1, FPR=1  
- Point `(0, 1)`: **perfect classifier** — catches all fraud with zero false alarms  
- The diagonal line from `(0,0)` to `(1,1)`: **random classifier** — equivalent to flipping a coin  

**AUC — Area Under the ROC Curve:**  
- AUC = 1.0 → perfect  
- AUC = 0.5 → random (useless)  
- AUC = 0.85+ → generally good for fraud detection  

**Probabilistic interpretation of AUC:**  
> AUC = probability that the model scores a randomly chosen fraud transaction **higher** than a randomly chosen legit transaction.

In [ ]:
# ── Cell 3: Imports and reproducibility setup ──────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    confusion_matrix, classification_report
)

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_palette('tab10')
SEED = 42
print('All libraries loaded successfully.')

In [ ]:
# ── Cell 4: Create imbalanced fraud dataset (5 000 samples, ~3% fraud) ─────
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=10,
    n_redundant=4,
    weights=[0.97, 0.03],   # 97% legit, 3% fraud
    flip_y=0.01,            # small label noise — realistic
    random_state=SEED
)

# Wrap in a DataFrame for clarity
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['fraud'] = y

# Summary
print('Dataset shape:', df.shape)
print('\nClass distribution:')
counts = df['fraud'].value_counts()
print(f'  Legit (0): {counts[0]:5d}  ({counts[0]/len(df)*100:.1f}%)')
print(f'  Fraud (1): {counts[1]:5d}  ({counts[1]/len(df)*100:.1f}%)')
print(f'\nImbalance ratio: {counts[0]/counts[1]:.1f}:1')

In [ ]:
# ── Cell 5: Train/test split and scaling ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

# Scale features (important for LogReg and k-NN)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Train four models ──────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'k-NN (k=5)':          KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB':         GaussianNB(),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=SEED)
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    fitted_models[name] = model
    print(f'Trained: {name}')

print('\nAll models trained on scaled features.')

## Building the ROC Curve from Scratch

The algorithm is straightforward:
1. Collect all unique prediction scores as candidate thresholds (sorted high → low).
2. For each threshold, apply it to convert scores into binary predictions.
3. Compute TPR and FPR from the resulting confusion matrix.
4. Append `(FPR, TPR)` to the curve.
5. Add the `(0, 0)` starting point and `(1, 1)` ending point.

The result is a set of points you can connect to draw the curve.

In [ ]:
# ── Cell 7: ROC curve implementation from scratch ──────────────────────────
def roc_curve_scratch(y_true, y_scores):
    """
    Compute ROC curve points by sweeping every unique score as a threshold.
    Returns (fpr_array, tpr_array, thresholds_array).
    """
    # Sort thresholds from highest to lowest (start predicting nothing)
    thresholds = np.sort(np.unique(y_scores))[::-1]

    # Anchor at (0, 0) — threshold above all scores → all predicted negative
    fprs, tprs = [0.0], [0.0]

    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)

        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        fn = int(np.sum((y_pred == 0) & (y_true == 1)))
        tn = int(np.sum((y_pred == 0) & (y_true == 0)))

        # TPR = recall; FPR = fall-out
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        fprs.append(fpr)
        tprs.append(tpr)

    # Anchor at (1, 1) — threshold = 0 → all predicted positive
    fprs.append(1.0)
    tprs.append(1.0)

    return np.array(fprs), np.array(tprs), thresholds

print('roc_curve_scratch() defined.')

In [ ]:
# ── Cell 8: Verify scratch ROC against sklearn for Logistic Regression ──────
logreg = fitted_models['Logistic Regression']
y_scores_lr = logreg.predict_proba(X_test_sc)[:, 1]  # fraud probability

# Scratch implementation
fpr_s, tpr_s, thresh_s = roc_curve_scratch(y_test, y_scores_lr)

# sklearn implementation
fpr_sk, tpr_sk, thresh_sk = roc_curve(y_test, y_scores_lr)

# Compare visually
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_sk, tpr_sk, lw=3, label='sklearn roc_curve', color='steelblue')
ax.plot(fpr_s,  tpr_s,  lw=2, linestyle='--', label='scratch implementation', color='tomato')
ax.plot([0, 1], [0, 1], 'k:', label='Random baseline')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR / Recall)')
ax.set_title('ROC Curve — Logistic Regression\n(scratch vs sklearn verification)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Number of points — scratch: {len(fpr_s)}, sklearn: {len(fpr_sk)}')
print('Curves match? The dashed line should overlap the solid line.')

## AUC — Area Under the Curve

**The trapezoidal rule** approximates the area under any curve by dividing it into trapezoids:

$$\text{AUC} \approx \sum_{i=1}^{n} \frac{(\text{FPR}_i - \text{FPR}_{i-1}) \times (\text{TPR}_i + \text{TPR}_{i-1})}{2}$$

Each term is the area of one trapezoid with:
- **width** = change in FPR (x-axis)
- **average height** = average of the two TPR values (y-axis)

Summing all trapezoids gives the total area under the curve.

In [ ]:
# ── Cell 10: Trapezoidal AUC from scratch ──────────────────────────────────
def auc_trapezoidal(fpr, tpr):
    """
    Compute AUC using the trapezoidal rule.
    fpr and tpr must be sorted by increasing fpr.
    """
    # Ensure sorted by FPR (ascending)
    order = np.argsort(fpr)
    fpr_sorted = fpr[order]
    tpr_sorted = tpr[order]

    auc = 0.0
    for i in range(1, len(fpr_sorted)):
        width   = fpr_sorted[i] - fpr_sorted[i - 1]          # delta FPR
        avg_height = (tpr_sorted[i] + tpr_sorted[i - 1]) / 2  # avg TPR
        auc += width * avg_height

    return auc

# Compare scratch AUC to sklearn for all four models
print(f'{'Model':<25} {'Scratch AUC':>12} {'sklearn AUC':>12} {'Diff':>8}')
print('-' * 62)
for name, model in fitted_models.items():
    scores = model.predict_proba(X_test_sc)[:, 1]
    fpr_s, tpr_s, _ = roc_curve_scratch(y_test, scores)
    auc_s  = auc_trapezoidal(fpr_s, tpr_s)
    auc_sk = roc_auc_score(y_test, scores)
    print(f'{name:<25} {auc_s:>12.4f} {auc_sk:>12.4f} {abs(auc_s-auc_sk):>8.6f}')

In [ ]:
# ── Cell 11: Plot all 4 ROC curves on one axes ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

for (name, model), color in zip(fitted_models.items(), colors):
    scores = model.predict_proba(X_test_sc)[:, 1]
    fpr_v, tpr_v, thresh_v = roc_curve(y_test, scores)  # use sklearn for clean curve
    auc_val = roc_auc_score(y_test, scores)

    ax.plot(fpr_v, tpr_v, lw=2, color=color,
            label=f'{name}  (AUC = {auc_val:.3f})')

    # Mark the default threshold=0.5 operating point
    # Find threshold index closest to 0.5
    idx = np.argmin(np.abs(thresh_v - 0.5))
    ax.scatter(fpr_v[idx], tpr_v[idx], s=80, color=color,
               marker='D', zorder=5)

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.500)')

# Annotate the diamond markers
ax.scatter([], [], s=80, marker='D', color='gray', label='Operating pt (thresh=0.5)')

ax.set_xlabel('False Positive Rate (FPR)\n← fewer false alarms')
ax.set_ylabel('True Positive Rate (TPR / Recall)\n← more fraud caught →')
ax.set_title('ROC Curves — Credit Card Fraud Detection\n(4 classifiers, 5 000 samples, 3% fraud)')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.05)
plt.tight_layout()
plt.show()

## Probabilistic Interpretation of AUC

> **AUC = P(score of a random fraud transaction > score of a random legit transaction)**

We can verify this empirically:
1. Sample many random **(fraud, legit)** pairs.
2. Count how many times the fraud score is higher than the legit score (ties count as 0.5).
3. The fraction should match AUC from the curve.

This interpretation is why AUC is often called the **rank-based** metric — it does not depend on the absolute scale of the scores, only on whether positives are ranked higher than negatives.

In [ ]:
# ── Cell 13: Probabilistic AUC interpretation (sampling verification) ───────
rng = np.random.default_rng(SEED)
N_PAIRS = 10_000   # number of (fraud, legit) pairs to sample

model_name = 'Logistic Regression'
scores_all = fitted_models[model_name].predict_proba(X_test_sc)[:, 1]

fraud_scores = scores_all[y_test == 1]
legit_scores  = scores_all[y_test == 0]

# Randomly sample N_PAIRS pairs and compare
fraud_sample = rng.choice(fraud_scores, size=N_PAIRS, replace=True)
legit_sample  = rng.choice(legit_scores,  size=N_PAIRS, replace=True)

# Count: fraud ranked higher (+1), tie (+0.5), legit higher (0)
wins  = np.sum(fraud_sample > legit_sample)
ties  = np.sum(fraud_sample == legit_sample)
empirical_auc = (wins + 0.5 * ties) / N_PAIRS

curve_auc = roc_auc_score(y_test, scores_all)

print(f'Model: {model_name}')
print(f'  Fraud scores  available: {len(fraud_scores)}')
print(f'  Legit scores  available: {len(legit_scores)}')
print(f'  Pairs sampled           : {N_PAIRS:,}')
print(f'  Wins (fraud > legit)    : {wins:,}')
print(f'  Ties                    : {ties}')
print(f'  Empirical AUC (sampling): {empirical_auc:.4f}')
print(f'  Curve AUC (sklearn)     : {curve_auc:.4f}')
print(f'  Difference              : {abs(empirical_auc - curve_auc):.4f}')
print('\n=> The numbers match closely — confirming the probabilistic interpretation.')

In [ ]:
# ── Cell 14: Partial AUC — compare models at FPR < 0.10 ────────────────────
# In fraud detection, operations teams cannot review many false alarms.
# So we care most about the TPR the model achieves at LOW FPR (say < 10%).
# Partial AUC (pAUC) measures only that left-hand slice of the ROC curve.

from sklearn.metrics import roc_auc_score

FPR_LIMIT = 0.10   # only care about FPR below 10%

def partial_auc(y_true, y_scores, max_fpr=0.10):
    """Return AUC restricted to FPR in [0, max_fpr], normalised to [0, 1]."""
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    # Keep only points with FPR <= max_fpr; clip last value to max_fpr
    mask = fpr <= max_fpr
    fpr_clip = np.append(fpr[mask], max_fpr)
    tpr_clip = np.append(tpr[mask], np.interp(max_fpr, fpr, tpr))
    raw_pauc = np.trapz(tpr_clip, fpr_clip)
    return raw_pauc / max_fpr   # normalise so perfect = 1.0

print(f'Partial AUC comparison (FPR < {FPR_LIMIT:.0%}):')
print(f'{'Model':<25} {'Full AUC':>10} {'pAUC(<10% FPR)':>16}')
print('-' * 55)
for name, model in fitted_models.items():
    scores = model.predict_proba(X_test_sc)[:, 1]
    full  = roc_auc_score(y_test, scores)
    pauc  = partial_auc(y_test, scores, max_fpr=FPR_LIMIT)
    print(f'{name:<25} {full:>10.4f} {pauc:>16.4f}')

print('\nNote: Ranking by partial AUC can differ from full AUC ranking.')
print('Choose pAUC when your operating region is constrained.')

In [ ]:
# ── Cell 15: Visualise Partial AUC region on the ROC plot ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: full ROC curves with shaded partial AUC region for best model
ax = axes[0]
best_model_name = 'Logistic Regression'
best_scores = fitted_models[best_model_name].predict_proba(X_test_sc)[:, 1]
fpr_b, tpr_b, _ = roc_curve(y_test, best_scores)

ax.plot(fpr_b, tpr_b, lw=2, color='steelblue', label=best_model_name)
ax.plot([0, 1], [0, 1], 'k--', lw=1)

# Shade partial AUC region (FPR <= 0.10)
mask = fpr_b <= FPR_LIMIT
ax.fill_between(fpr_b[mask], tpr_b[mask], alpha=0.25, color='steelblue',
                label=f'Partial AUC (FPR<{FPR_LIMIT:.0%})')
ax.axvline(FPR_LIMIT, color='tomato', linestyle=':', lw=1.5,
           label='FPR = 10% boundary')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Partial AUC — Low False Alarm Zone')
ax.legend(fontsize=9)

# Right: bar chart of pAUC for all models
ax2 = axes[1]
paucs = {}
for name, model in fitted_models.items():
    sc = model.predict_proba(X_test_sc)[:, 1]
    paucs[name] = partial_auc(y_test, sc, max_fpr=FPR_LIMIT)

bars = ax2.barh(list(paucs.keys()), list(paucs.values()),
                color=plt.cm.tab10(np.linspace(0, 0.4, 4)))
ax2.set_xlabel(f'Normalised Partial AUC (FPR < {FPR_LIMIT:.0%})')
ax2.set_title('Which model is best at low false-alarm rates?')
for bar, val in zip(bars, paucs.values()):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Warning: AUC Can Be Misleading on Extremely Imbalanced Data

Consider a dataset where **99% of transactions are legit** and only 1% are fraud.

Even a model that is *mediocre* at finding fraud can look impressive on ROC because:
- The denominator of FPR = `FP / (FP + TN)` is dominated by the huge number of true negatives (TN).
- So FPR stays low even when the model makes many mistakes on the majority class.
- This inflates the AUC artificially.

The next notebook (Notebook 8) covers the **Precision-Recall curve** which avoids this problem by never involving true negatives in its formulas.

In [ ]:
# ── Cell 17: Demonstrate misleading AUC on 99:1 imbalanced data ────────────
# Create a severely imbalanced dataset (99% legit, 1% fraud)
X_imb, y_imb = make_classification(
    n_samples=5000, n_features=20, n_informative=5,
    weights=[0.99, 0.01], flip_y=0.01, random_state=SEED
)
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=SEED, stratify=y_imb
)
sc_i = StandardScaler()
X_tr_i = sc_i.fit_transform(X_tr_i)
X_te_i = sc_i.transform(X_te_i)

lr_imb = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_tr_i, y_tr_i)
scores_imb = lr_imb.predict_proba(X_te_i)[:, 1]

auc_imb = roc_auc_score(y_te_i, scores_imb)
base_rate = y_te_i.mean()

print(f'Dataset: 5 000 samples, {base_rate*100:.1f}% fraud')
print(f'Fraud transactions in test set: {y_te_i.sum()}')
print(f'AUC-ROC: {auc_imb:.4f}   <- Looks great!')
print()

# But look at what happens at threshold = 0.5 (default)
y_pred_default = (scores_imb >= 0.5).astype(int)
cm = confusion_matrix(y_te_i, y_pred_default)
print('Confusion matrix at threshold = 0.5:')
print(cm)
detected = cm[1, 1]
missed   = cm[1, 0]
print(f'Fraud CAUGHT: {detected} / {detected+missed}  ({detected/(detected+missed)*100:.1f}%)')
print(f'Fraud MISSED: {missed}   <- the real story!')
print('\n=> AUC=0.97 can hide poor recall. Always inspect the PR curve too.')

In [ ]:
# ── Cell 18: Visual summary — score distributions for fraud vs legit ────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: score distributions
ax = axes[0]
scores_main = fitted_models['Logistic Regression'].predict_proba(X_test_sc)[:, 1]
ax.hist(scores_main[y_test == 0], bins=40, alpha=0.6, color='steelblue',
        density=True, label='Legit transactions')
ax.hist(scores_main[y_test == 1], bins=20, alpha=0.7, color='tomato',
        density=True, label='Fraud transactions')
ax.axvline(0.5, color='black', linestyle='--', label='Default threshold (0.5)')
ax.set_xlabel('Predicted fraud probability')
ax.set_ylabel('Density')
ax.set_title('Score Distributions — Logistic Regression\n(3% fraud rate)')
ax.legend(fontsize=9)

# Right: ROC curve with shaded AUC
ax2 = axes[1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, scores_main)
auc_lr = roc_auc_score(y_test, scores_main)
ax2.fill_between(fpr_lr, tpr_lr, alpha=0.15, color='steelblue')
ax2.plot(fpr_lr, tpr_lr, lw=2, color='steelblue',
         label=f'Logistic Regression (AUC={auc_lr:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', label='Random baseline')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve with Shaded AUC')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 19: Comprehensive model comparison dashboard ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

for idx, ((name, model), color) in enumerate(zip(fitted_models.items(), colors)):
    ax = axes[idx]
    scores = model.predict_proba(X_test_sc)[:, 1]
    fpr_v, tpr_v, thresh_v = roc_curve(y_test, scores)
    auc_val = roc_auc_score(y_test, scores)
    pauc_val = partial_auc(y_test, scores, max_fpr=0.10)

    ax.fill_between(fpr_v, tpr_v, alpha=0.15, color=color)
    ax.plot(fpr_v, tpr_v, lw=2, color=color)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.axvline(0.10, color='grey', linestyle=':', lw=1)

    # Mark threshold=0.5
    idx_op = np.argmin(np.abs(thresh_v - 0.5))
    ax.scatter(fpr_v[idx_op], tpr_v[idx_op], s=100, color='black',
               marker='D', zorder=5, label='thresh=0.5')

    ax.set_title(f'{name}\nAUC={auc_val:.3f}  |  pAUC(<10%)={pauc_val:.3f}')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=8)
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)

plt.suptitle('Individual ROC Curves — Credit Card Fraud Detection', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | One-sentence summary |
|---|---|
| ROC curve | Plots every (FPR, TPR) tradeoff as the threshold sweeps 0→1 |
| AUC | Area under ROC; 0.5 = random, 1.0 = perfect |
| Probabilistic AUC | P(fraud score > legit score) for a random pair |
| Partial AUC | Restricts comparison to operationally relevant FPR range |
| AUC limitation | Can be optimistic on severely imbalanced data — always pair with PR curve |

**When to use ROC/AUC:**
- Classes are balanced or mildly imbalanced
- You need a single threshold-independent metric for model selection
- You want to compare the ranking quality of models

**When NOT to rely solely on AUC:**
- Severe class imbalance (< 5% positives) → use PR curve
- You have a fixed operating point → report TPR at that FPR directly
- FP and FN have very different costs → use cost-sensitive metrics

## Self-Check Questions

Test your understanding. Try to answer each question before expanding the answer.

---

**Question 1:**  
AUC = P(score of random positive > score of random negative). If AUC = 0.5, what does this mean?

<details>
<summary>Show Answer</summary>

AUC = 0.5 means the model cannot distinguish fraud from legit transactions at all. For any randomly chosen fraud/legit pair, the model ranks the fraud higher only 50% of the time — the same as flipping a fair coin. This is equivalent to a random classifier. The ROC curve would be the diagonal line from (0,0) to (1,1). The model has learned nothing useful from the data.

</details>

---

**Question 2:**  
Model A has AUC = 0.92, Model B has AUC = 0.88. But at FPR = 0.05 (your operating point), Model B has higher TPR. Which model do you choose?

<details>
<summary>Show Answer</summary>

You should choose **Model B** for deployment. AUC is a global, threshold-independent summary that averages performance across ALL operating points. But you have told us your operating point: FPR = 0.05. At that specific point, Model B catches more fraud (higher TPR) for the same false-alarm rate. Global AUC is irrelevant here — what matters is performance at YOUR operating point. This is precisely why partial AUC (measured only in the FPR range you care about) is often more useful than full AUC for real deployment decisions.

</details>

---

**Question 3:**  
Your fraud model has AUC = 0.97 on a 99:1 dataset. A colleague says "great model!" What should you check next?

<details>
<summary>Show Answer</summary>

Several things deserve scrutiny:

1. **Precision-Recall curve and Average Precision (AP):** With 1% fraud, a high AUC can be achieved even if the model barely finds any fraud (because the huge TN pool keeps FPR low). AP directly measures precision vs recall on the minority class — this is the harder, more honest metric.
2. **Recall at your operating threshold:** At threshold = 0.5, how many actual fraud cases does the model catch? As shown in Cell 17, a model with AUC = 0.97 might still miss most fraud at the default threshold.
3. **Confusion matrix at the deployment threshold:** How many FPs will operations need to review? How many FNs (missed fraud) will slip through?
4. **Cost analysis:** In fraud, a missed fraud case (FN) typically costs far more than a false alarm (FP). Use the business cost function, not just AUC.

Short answer: check the **PR curve / AP score** and the confusion matrix at your operating threshold before declaring victory.

</details>

## Next Steps

- **Notebook 8:** Precision-Recall Curve & Average Precision — the companion metric for imbalanced data
- **Notebook 9:** Cost-sensitive evaluation — when FP and FN have different dollar values
- **Notebook 10:** Calibration — are those predicted probabilities actually reliable?

---
*Week 7 — Model Evaluation, Validation & Metrics*